# Solar Filament Segmentation: train on Kaggle and submit

End-to-end pipeline on Kaggle GPUs: **setup → train the 5 folds → tune post-processing on out-of-fold predictions →
predict the test set → write and validate `submission.csv`.**

All logic lives in the repository ([LucaT2/solar-filament-segmentation](https://github.com/LucaT2/solar-filament-segmentation)),
so this notebook only orchestrates. The same commands run locally (see the README).

**Notebook settings:** Accelerator **GPU T4 ×2** (two folds train in parallel, one per GPU), Internet **on**,
competition data **filament-segmentation-2026** attached.

**Code access** (the repo is private until the deadline), either:
* **Option A:** add a Kaggle *Secret* named `GITHUB_TOKEN` (a GitHub fine-grained token with read access to the
  repo; *Add-ons → Secrets*), or
* **Option B:** upload the repo as a Kaggle Dataset and attach it (set `REPO_DATASET` below).

In [ ]:
# ---- configuration ---------------------------------------------------------------------------------------
CONFIG = "configs/unet_r34.yaml"   # experiment config inside the repo
OVERRIDES = []                      # e.g. ["epochs=20", "num_workers=2"]
FOLDS = [0, 1, 2, 3, 4]             # folds to train in this session ([] = skip training, use attached checkpoints)
TTA = True                          # average 4 flips at test time
REPO_URL = "github.com/LucaT2/solar-filament-segmentation.git"
REPO_BRANCH = "main"
REPO_DATASET = None                 # Option B: e.g. "/kaggle/input/solar-filament-code"
CHECKPOINT_DATASET = None           # optional: dataset with previously trained fold*.pt + oof/ to reuse

## 1. Setup: code and packages

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

REPO = Path("/kaggle/working/repo")
if REPO_DATASET:
    shutil.copytree(REPO_DATASET, REPO, dirs_exist_ok=True)
elif not REPO.exists():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, f"https://{token}@{REPO_URL}", str(REPO)],
                   check=True, capture_output=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# Kaggle images ship torch, numpy, pandas, opencv, scikit-image and timm. Install only what is missing, pinned
# to the versions tested locally (requirements.txt), without touching the preinstalled stack.
!pip install -q --no-deps segmentation-models-pytorch==0.5.0 albumentations==2.0.8 albucore==0.0.24 pycocotools
!pip install -q --no-deps -e {REPO}
sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

import torch
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
!python scripts/check_env.py

## 2. Train the folds

Each fold trains on the other 4 folds, keeps the checkpoint with the best validation PQ, and writes probability
maps for its validation images to `outputs/<name>/oof/`. Folds are distributed over the available GPUs. A fold takes
roughly 45–60 min on a T4 with the baseline config, so 5 folds on 2 GPUs take about 3 hours.

In [ ]:
from filament.train import TrainConfig
cfg = TrainConfig.load(CONFIG, OVERRIDES)
EXP = Path("/kaggle/working") / cfg.name
print(cfg)

if CHECKPOINT_DATASET:   # reuse earlier sessions' results
    shutil.copytree(CHECKPOINT_DATASET, EXP, dirs_exist_ok=True)

def train_folds_on_gpus(folds):
    n_gpu = max(torch.cuda.device_count(), 1)
    queues = [folds[i::n_gpu] for i in range(n_gpu)]
    procs = []
    for gpu, fold_list in enumerate(queues):
        if not fold_list:
            continue
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
        cmd = [sys.executable, "-W", "ignore", "scripts/train.py", "--config", CONFIG,
               "--fold", *map(str, fold_list), "--set", *OVERRIDES]
        log = open(f"/kaggle/working/train_gpu{gpu}.log", "w")
        procs.append((subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT), log, gpu))
        print(f"GPU {gpu}: folds {fold_list}")
    for p, log, gpu in procs:
        p.wait(); log.close()
        print(f"GPU {gpu} finished with exit code {p.returncode}")
        print("".join(open(f"/kaggle/working/train_gpu{gpu}.log").readlines()[-6:]))

if FOLDS:
    train_folds_on_gpus(FOLDS)

In [ ]:
# Training curves: validation PQ at each threshold per epoch
import pandas as pd
for f in sorted(EXP.glob("log_fold*.csv")):
    log = pd.read_csv(f)
    pq_cols = [c for c in log.columns if c.startswith("pq@")]
    best = log[pq_cols].max(axis=1).max()
    print(f"{f.stem}: {len(log)} epochs, best val PQ {best:.4f}, last loss {log.loss.iloc[-1]:.4f}")

## 3. Tune post-processing on out-of-fold predictions

Grid search over the probability threshold (outline width), minimum component area and minimum mean probability,
scored with the exact competition PQ on all out-of-fold images. The best setting is saved to `postprocess.json`.

In [ ]:
!python -W ignore scripts/tune_postprocess.py --exp {cfg.name} --workers 4 \
    --thresholds 0.25 0.3 0.35 0.4 0.45 0.5 0.55 0.6 --min-areas 0 25 50 100 150 200 300 --min-scores 0 0.5 0.6 0.7

In [ ]:
# Cross-check: write the OOF predictions as a CSV and score it with the verified metric implementation
!python -W ignore scripts/make_submission.py --exp {cfg.name} --split oof
!python -W ignore scripts/evaluate.py --pred /kaggle/working/{cfg.name}/oof_submission.csv

## 4. Predict the test set and write the submission

In [ ]:
tta_flag = "--tta" if TTA else ""
!python -W ignore scripts/predict.py --exp {cfg.name} {tta_flag}
!python -W ignore scripts/make_submission.py --exp {cfg.name}
shutil.copy(EXP / "submission.csv", "/kaggle/working/submission.csv")

In [ ]:
from filament.submission import read_submission, validate_submission
sub = read_submission("/kaggle/working/submission.csv")
test_stems = [p.stem for p in (Path(os.environ.get("FILAMENT_DATA", "")) if os.environ.get("FILAMENT_DATA")
              else __import__("filament.paths", fromlist=["x"]).get_data_paths().test_images).glob("*.jpeg")]
print(len(sub), "filaments,", sub.filament_id.str.rsplit("_", n=1).str[0].nunique(), "of", len(test_stems), "images")
print("validation:", validate_submission(sub, expected_stems=test_stems) or "OK")
sub.head()

## 5. Submit

`/kaggle/working/submission.csv` is in this notebook's output. Download it and upload it on the competition page
(*Submit Prediction*), or from a machine with Kaggle credentials:

```
kaggle competitions submit -c filament-segmentation-2026 -f submission.csv -m "unet_r34 5-fold + TTA"
```

To reuse the trained models in a later session, save this notebook's output as a Dataset and set
`CHECKPOINT_DATASET` (with `FOLDS = []` to skip training).